# XP Exercises: Flower Classification using CNN

This is a guided notebook for the exercises on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear only for key concepts that unlock intuition or transfer to other ML topics.


## What you will learn
- Building a CNN for multi class image classification
- Data loading and preprocessing with `image_dataset_from_directory`
- Image visualization techniques
- Model architecture design, compilation, and training
- Evaluating model performance with accuracy and loss plots


## What you will create
A CNN model that classifies 14 flower species.
All parts form one continuous exercise. Work through them sequentially.


## Dataset
**As stated in the exercises**  
Flower classification with 14 classes. Images are organized in class folders. A training and validation split may be provided. Images are resized to 256x256 in this notebook.

**PREFILLED info**  
This notebook expects the provided zip file to be available. The code below extracts it and locates the dataset root automatically.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# PREFILLED: just execute
import os, sys, zipfile, shutil, glob, math, json, random
from pathlib import Path

DATA_ZIP = Path("./Flower Classification.zip")
EXTRACT_DIR = Path("./data/flower_data")

# Clean extract dir if re-running
if EXTRACT_DIR.exists():
    pass  # avoid deleting in case you added files; delete manually if needed
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract if a zip is present and not already extracted
if DATA_ZIP.exists():
    # Heuristically decide to extract once
    marker = EXTRACT_DIR / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        marker.write_text("ok")
        print("Extracted:", DATA_ZIP.name, "->", EXTRACT_DIR)
    else:
        print("Already extracted. Skipping.")
else:
    print("Zip file not found at", DATA_ZIP)

# Find candidate dataset roots: a dir with >= 10 subdirs assumed as classes, or contains train/val
def list_dirs(p):
    return [d for d in Path(p).iterdir() if d.is_dir()]

candidates = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    if len([d for d in Path(root).iterdir() if Path(d).is_dir()]) >= 10:
        candidates.append(Path(root))
    if "train" in [d.name.lower() for d in list_dirs(root)] and "val" in [d.name.lower() for d in list_dirs(root)]:
        candidates.append(Path(root))

candidates = sorted(set(candidates))
print("Candidate dataset roots:", [str(c) for c in candidates][:5])

Zip file not found at Flower Classification.zip
Candidate dataset roots: []


## Part 1. Data exploration and visualization



In [ ]:
# PREFILLED: just execute
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (32, 32)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def detect_layout(root: Path):
    root = Path(root)
    sub = [d.name.lower() for d in root.iterdir() if d.is_dir()]
    if "train" in sub and "val" in sub:
        return "provided_split", root
    return "single_root", root

# Choose a root
if 'candidates' in globals() and len(candidates) > 0:
    DS_ROOT = candidates[0]
else:
    DS_ROOT = EXTRACT_DIR  # fallback

layout, base = detect_layout(DS_ROOT)
print("Layout:", layout, "Base:", base)

In [ ]:
# PREFILLED: just execute
if layout == "provided_split":
    train_dir = next((p for p in base.iterdir() if p.name.lower()=="train"))
    val_dir   = next((p for p in base.iterdir() if p.name.lower()=="val"))
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", num_classes, class_names)

# Cache and prefetch
def prepare(ds):
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prepare(train_ds)
val_ds = prepare(val_ds)

In [ ]:
# PREFILLED: just execute — count images per class by scanning directory
from collections import Counter
import os

def count_images_per_class(root):
    counts = {}
    for cls in class_names:
        # find folder named like cls at any depth under base
        matches = list(Path(base).rglob(cls))
        if matches:
            folder = matches[0]
            img_count = sum(1 for p in folder.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".gif"})
            counts[cls] = img_count
        else:
            counts[cls] = None
    return counts

base = "/content/data/flower_data/Data/train"
counts = count_images_per_class(base)
counts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_images(dataset, class_names, per_class=9):
    """
    Display a 3x3 grid of images for each class in class_names.

    Args:
        dataset: a tf.data.Dataset object yielding (image, label) batches
        class_names: list of class names (strings)
        per_class: number of images to show per class (default 9)
    """
    # Collect images by class
    collected = {cls: [] for cls in class_names}

    for images, labels in dataset:
        for img, lbl in zip(images, labels):
            cls = class_names[int(lbl)]
            if len(collected[cls]) < per_class:
                collected[cls].append(img.numpy())
        # Stop early if all classes have enough
        if all(len(collected[cls]) >= per_class for cls in class_names):
            break

    # Plot each class separately
    for cls in class_names:
        imgs = collected[cls][:per_class]
        plt.figure(figsize=(6,6))
        for i, img in enumerate(imgs):
            plt.subplot(3,3,i+1)
            plt.imshow(img.astype("uint8"))
            plt.axis("off")
        plt.suptitle(cls)
        plt.show()

Flower classification is challenging because many species share similar colors, making them hard to distinguish. Variations in lighting, background, and bloom stage add noise, and class imbalance can cause the model to favor dominant classes.

## Part 2. Model architecture design


In [ ]:
# PREFILLED: just execute — baseline model scaffold
from tensorflow.keras import models

def build_baseline(num_classes):
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),  # safety if datasets were not normalized
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_baseline(num_classes)
baseline.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_variant(num_classes):
    model = models.Sequential()

    # First block: larger kernel
    model.add(layers.Conv2D(32, (5,5), activation='relu', input_shape=(224,224,3)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    # Second block
    model.add(layers.Conv2D(64, (5,5), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    # Third block
    model.add(layers.Conv2D(128, (3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    # Fourth block
    model.add(layers.Conv2D(256, (3,3), activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    # Flatten + Dense layers
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.4))

    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

# Build and summarize
model_variant = build_variant(num_classes)
model_variant.summary()

The model increases filters progressively (32→64→128→256) to expand the receptive field and capture more complex features at deeper layers. BatchNormalization after convolution and dense layers stabilizes training by reducing internal covariate shift. Using larger kernels (5×5) early helps capture broader spatial patterns, while Dropout at 0.4 provides regularization to reduce overfitting and improve generalization.

## Part 3. Hyperparameter tuning

**As stated in the exercises**  
Experiment with optimizers, learning rate, batch size, and optionally learning rate scheduling or early stopping. Track experiments and results. Report the best combination.


In [ ]:
# PREFILLED: just execute — utilities for training and plotting
import time

def fit_model(model, train_ds, val_ds, epochs=5, callbacks=None):
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)
    dt = time.time() - t0
    return history, dt

def plot_curves(history, title="Training"):
    plt.figure(figsize=(6,4))
    plt.plot(history.history.get("accuracy", []), label="acc")
    plt.plot(history.history.get("val_accuracy", []), label="val_acc")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(6,4))
    plt.plot(history.history.get("loss", []), label="loss")
    plt.plot(history.history.get("val_loss", []), label="val_loss")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
import tensorflow as tf

# Define search space
opts = [
    ("adam", 1e-3, 32),
    ("adam", 5e-4, 32),
    ("rmsprop", 1e-3, 32),
    ("sgd", 1e-2, 64),
]

results = []
for opt_name, lr, batch in opts:
    # Rebuild model each time (baseline or variant)
    model = build_baseline(num_classes)  # or build_variant(num_classes)

    # Select optimizer
    if opt_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    elif opt_name == "rmsprop":
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True)

    # Compile
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    # Early stopping
    cb = [tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]

    # Fit model
    hist, dur = fit_model(model, train_ds, val_ds, epochs=8, callbacks=cb)

    # Record best validation accuracy
    best_val = max(hist.history["val_accuracy"])
    results.append({
        "opt": opt_name,
        "lr": lr,
        "batch": batch,
        "best_val_acc": float(best_val),
        "time_s": round(dur, 1)
    })

results

The best hyperparameters were Adam with a learning rate of 1e‑3 and batch size 32, which gave the highest validation accuracy. This works well because Adam adapts learning rates per parameter, providing stable and efficient convergence for a dataset with varied flower classes.


## Part 4. Data augmentation


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path

# Locate train directory
train_dir = next((p for p in Path(base).iterdir() if p.name.lower()=='train'), None)
if train_dir is None:
    train_dir = base  # fallback if only one root

# Define augmentation pipeline
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

# Training and validation flows
flow_train = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training',
    seed=SEED
)

flow_val = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation',
    seed=SEED
)

# Train with augmentation
model_aug = build_baseline(num_classes)  # or use your variant model
hist_aug = model_aug.fit(
    flow_train,
    validation_data=flow_val,
    epochs=8,
    verbose=2
)

plot_curves(hist_aug, title='Augmented training')

## Part 5. Performance evaluation and analysis

In [ ]:
# PREFILLED: just execute — helpers for evaluation on a dataset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def collect_preds(model, ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pr = model.predict(xb, verbose=0)
        y_prob.append(pr)
        y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    if y_prob.ndim == 2 and y_prob.shape[1] > 1:
        y_pred = y_prob.argmax(axis=1)
    else:
        y_pred = (y_prob.ravel() >= 0.5).astype(int)
    return y_true, y_pred, y_prob

def plot_confusion(cm, labels):
    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title("Confusion matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=90)
    plt.yticks(ticks, labels)
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Choose your best model (baseline, variant, or augmented)
best_model = model_aug  # or model_variant, depending on your results

# Collect predictions
y_true, y_pred, y_prob = collect_preds(best_model, val_ds)

# Classification report
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plot_confusion(cm, class_names)

In [ ]:
import random
import matplotlib.pyplot as plt

# Number of samples to visualize
take = 12

# Grab a batch of validation images
imgs, labels = next(iter(val_ds.unbatch().batch(take)))

# Get predictions
probs = best_model.predict(imgs, verbose=0)
preds = probs.argmax(axis=1)

# Plot each image with true vs predicted label
for i in range(take):
    plt.figure(figsize=(2.5,2.5))
    plt.imshow(imgs[i].numpy().astype('uint8'))
    t = f"true: {class_names[int(labels[i])]}\npred: {class_names[int(preds[i])]}"
    plt.title(t)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

The model struggled most with classes that share similar morphology or color palettes, such as daisies and asters or roses and tulips. These visual overlaps make it harder for the network to distinguish fine details, and small sample counts in certain classes further increase misclassification risk.


## Part 6. Model saving and deployment (optional)

In [ ]:
# Save in TensorFlow SavedModel format (directory)
best_model.save("./data/flower_cnn_savedmodel")

# Save in HDF5 format (single file)
best_model.save("./data/flower_cnn.h5")

print("Saved to ./data/")